# MongoDB and Web Mining

**DSC 785 Text Analytics, Adelphi University (2024 offering)**

This notebook covers two topics:

1. **ETF holdings into MongoDB.** Download the SPDR product list and the daily holdings files from ssga.com, compare three ways of turning a holdings table into documents, store one document per ETF, then add indexes and read `explain()` to see whether a query uses them.
2. **Web mining with Beautiful Soup.** Scrape the ETF list on stockanalysis.com twice: once from the HTML `<table>`, and once from the JSON embedded in a `<script type='application/json'>` tag.

To run it, put your own MongoDB Atlas connection string in the connection cell. Outputs that listed individual holdings (with CUSIP and SEDOL identifiers) were cleared; the notebook downloads the current files itself when run.


Product Data

In [1]:
!wget https://www.ssga.com/us/en/intermediary/etfs/library-content/products/fund-data/etfs/us/spdr-product-data-us-en.xlsx

--2024-06-17 12:44:48--  https://www.ssga.com/us/en/intermediary/etfs/library-content/products/fund-data/etfs/us/spdr-product-data-us-en.xlsx
Resolving www.ssga.com (www.ssga.com)... 23.210.30.20, 2600:1408:9000:784::d7f, 2600:1408:9000:79f::d7f
Connecting to www.ssga.com (www.ssga.com)|23.210.30.20|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.ssga.com/library-content/products/fund-data/etfs/us/spdr-product-data-us-en.xlsx [following]
--2024-06-17 12:44:48--  https://www.ssga.com/library-content/products/fund-data/etfs/us/spdr-product-data-us-en.xlsx
Reusing existing connection to www.ssga.com:443.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/vnd.openxmlformats-officedocument.spreadsheetml.sheet]
Saving to: ‘spdr-product-data-us-en.xlsx’

spdr-product-data-u     [ <=>                ]  75.47K  --.-KB/s    in 0.02s   

2024-06-17 12:44:48 (3.84 MB/s) - ‘spdr-product-data-us-en.xlsx’ saved [77280

In [2]:
import pandas as pd
etf_df=pd.read_excel('spdr-product-data-us-en.xlsx',skiprows=1).dropna()
etf_tickers=etf_df.loc[etf_df['Name'].str.contains('(?i)select sector')].Ticker.to_list()


Holdings Data

In [3]:
! wget https://www.ssga.com/us/en/intermediary/etfs/library-content/products/fund-data/etfs/us/us_spdrallholdings.zip

--2024-06-17 12:44:51--  https://www.ssga.com/us/en/intermediary/etfs/library-content/products/fund-data/etfs/us/us_spdrallholdings.zip
Resolving www.ssga.com (www.ssga.com)... 23.210.30.20, 2600:1408:9000:784::d7f, 2600:1408:9000:79f::d7f
Connecting to www.ssga.com (www.ssga.com)|23.210.30.20|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.ssga.com/library-content/products/fund-data/etfs/us/us_spdrallholdings.zip [following]
--2024-06-17 12:44:51--  https://www.ssga.com/library-content/products/fund-data/etfs/us/us_spdrallholdings.zip
Reusing existing connection to www.ssga.com:443.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘us_spdrallholdings.zip’

us_spdrallholdings.     [ <=>                ]   7.55M  38.4MB/s    in 0.2s    

2024-06-17 12:44:52 (38.4 MB/s) - ‘us_spdrallholdings.zip’ saved [7912717]



In [4]:
! unzip *zip

Archive:  us_spdrallholdings.zip
  inflating: holdings-daily-us-en-hybl.xlsx  
  inflating: holdings-daily-us-en-srln.xlsx  
  inflating: holdings-daily-us-en-tipx.xlsx  
  inflating: holdings-daily-us-en-bil.xlsx  
  inflating: holdings-daily-us-en-bils.xlsx  
  inflating: holdings-daily-us-en-cwb.xlsx  
  inflating: holdings-daily-us-en-ebnd.xlsx  
  inflating: holdings-daily-us-en-emhc.xlsx  
  inflating: holdings-daily-us-en-jnk.xlsx  
  inflating: holdings-daily-us-en-ibnd.xlsx  
  inflating: holdings-daily-us-en-bwx.xlsx  
  inflating: holdings-daily-us-en-flrn.xlsx  
  inflating: holdings-daily-us-en-sjnk.xlsx  
  inflating: holdings-daily-us-en-bwz.xlsx  
  inflating: holdings-daily-us-en-emtl.xlsx  
  inflating: holdings-daily-us-en-stot.xlsx  
  inflating: holdings-daily-us-en-totl.xlsx  
  inflating: holdings-daily-us-en-rwo.xlsx  
  inflating: holdings-daily-us-en-dia.xlsx  
  inflating: holdings-daily-us-en-rwx.xlsx  
  inflating: holdings-daily-us-en-rwr.xlsx  
  inflatin

In [5]:
!ls holdings-daily-us-en-spy.xlsx

holdings-daily-us-en-spy.xlsx


In [6]:
import pandas as pd
spy_df = pd.read_excel('holdings-daily-us-en-spy.xlsx', skiprows=4)
spy_df = spy_df.dropna()

In [7]:
spy_df

In [8]:
# Hard to access, but one document
d=spy_df.head(3).to_dict()


In [9]:
# Easier to access but 500 documents
spy_df.head(3).to_dict('records')

In [10]:
# Easier to access and one document
doc = {
    'etf':'spy',
    'holdings':spy_df.head(3).to_dict('records')
}
doc


In [ ]:
!pip install pymongo
from pymongo import MongoClient
from datetime import datetime

# Provide the mongodb atlas url to connect python to mongodb using pymongo
CONNECTION_STRING = 'mongodb+srv://<your_user>:<your_password>@<your-cluster>.mongodb.net/'   # your own Atlas cluster

# Create a connection using MongoClient. You can import MongoClient or use pymongo.MongoClient
client = MongoClient(CONNECTION_STRING)

# Database and Collection
db_name = 'testDB'
collection_name = 'testETFs'

# Access the database and collection
db = client[db_name]
collection = db[collection_name]



In [ ]:
collection.delete_many({})

In [ ]:
for etf_tic in etf_tickers+['SPY']:
  spy_df = pd.read_excel(f'holdings-daily-us-en-{etf_tic.lower()}.xlsx', skiprows=4)
  spy_df = spy_df.dropna()

  # Easier to access and one document
  doc = {
      'date':f'{datetime.now():%Y%m%d}',
      'etf':etf_tic,
      'holdings':spy_df.to_dict('records')
  }
  insert_result = collection.insert_one(doc)

In [ ]:
pd.DataFrame(collection.find({}))

In [ ]:
collection.create_index([('etf',1)])

In [ ]:
collection.find({'etf':'XLK'}).explain()

In [ ]:
pd.DataFrame(pd.DataFrame(collection.find({'etf':'XLK'})).sort_values('date').iloc[-1].holdings) #latest holdings by date

In [ ]:
pd.DataFrame(collection.find({'holdings.Ticker':'NVDA'}))

In [ ]:
collection.find({'holdings.Ticker':'NVDA'}).explain()

In [ ]:
collection.create_index([('holdings.Ticker',1)])

In [ ]:
from bs4 import BeautifulSoup
import requests

# Get the HTML
res = requests.get('https://stockanalysis.com/etf/')
if res.status_code>200:
  print('Error')


In [ ]:
# Parse HTML
soup = BeautifulSoup(res.text, 'html.parser')


In [ ]:
len(soup.find_all('table')) # There is one table

In [ ]:
firsttable = soup.find('table')
firsttable.find_all('tr')[0]

In [ ]:
firsttable.find_all('tr')[1]

In [ ]:
for th in firsttable.find_all('tr')[0].find_all('th'):
  print(th.text)

In [ ]:
for th in firsttable.find_all('tr')[1].find_all('td'):
  print(th.text)

In [ ]:
'''
<html>
  <body>
      <table>
          <tr> <th> row1_head1 </th>  <th> row1_head2 </th> </tr>
          <tr> <td> row1_cell1 </td>  <td> row1_cell2 </td> </tr>
          <tr> <td> row2_cell1 </td>  <td> row2_cell2 </td> </tr>
      </table>
  </body>
</html>
'''

In [ ]:
header_names = [th.text for th in firsttable.find_all('tr')[0].find_all('th')]


In [ ]:
data=[]
for row in firsttable.find_all('tr')[1:]:
  data.append([td.text for td in row.find_all('td')])
data[:3]

In [ ]:
data=[[td.text for td in row.find_all('td')] for row in firsttable.find_all('tr')[1:]]

In [ ]:
pd.DataFrame(data,columns=header_names)

In [ ]:
# Alternatively get the json within <script type='application/json' ....

len(soup.find_all('script')) # 4 matches

In [ ]:
len(soup.find_all('script',{'type':'application/json'})) # 1 matches

In [ ]:
script_text = soup.find_all('script',{'type':'application/json'})[0].text
script_text

In [ ]:
import regex
data_string = regex.findall(r'\[(.*?)\]',script_text)[0] # regular expression to just get the contents [****] of {...,"data":[****]}
# Clean \\" as "
data_string=data_string.replace('\\"','"')
# define null as nan
import numpy as np
null = np.nan

data_dict=eval(data_string)

In [ ]:
all_etfs = pd.DataFrame(data_dict).rename(columns={'s':'ticker','n':'name','i':'asset_class','m':'market_cap'})
all_etfs

In [ ]:
f = all_etfs.asset_class=='Equity'
all_etfs.loc[f,]